# Reentrenamiento Top-5 combinaciones Optuna

Carga las 5 mejores combinaciones de hiperparámetros del estudio Optuna  
y las reentrena con más épocas para obtener el mejor resultado posible de cada una.

**Estudio**: `mdlf_marzo2025_v2`  
**Referencia**: `mdlf_params.ipynb`

In [ ]:
import os, sys, json
from pathlib import Path

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f'CWD -> {os.getcwd()}')

import optuna
import pandas as pd
from datetime import datetime
from pipeline_mdlf import PipelineMDLF

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────

STUDY_NAME   = 'mdlf_marzo2025_v2'
ALMACEN      = '50mil_tarjetas_3_al_31_marzo'
SPLIT_DATE   = '2025-03-24'
N_TOP        = 5      # número de combinaciones a reentrenar
N_EPOCHS     = 60     # épocas por reentrenamiento — ajusta según tiempo disponible

DB_PATH      = _mdlf_dir / 'runs' / f'{STUDY_NAME}.db'
LINE_AP_PATH = _mdlf_dir / 'LINE' / 'output' / 'line_cercanias_optuna.parquet'

_descomp = Path('/home/jupyter-lquijano/david/estudios/descomprimido')
TTP_PATHS = [
    str(_descomp / 'TTP_PERIODO_01_2025.csv'),
    str(_descomp / 'TTP_PERIODO_02_2025.csv'),
    str(_descomp / 'TTP_PERIODO_03_2025.csv'),
]
OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

print(f'Estudio  : {STUDY_NAME}')
print(f'DB       : {DB_PATH}')
print(f'Top-N    : {N_TOP} combinaciones')
print(f'Épocas   : {N_EPOCHS}')

In [ ]:
# ── Cargar estudio y extraer top-N trials ─────────────────────────────────────

study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{DB_PATH}',
)

df_trials = study.trials_dataframe()
df_ok = df_trials[df_trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)

top_trials = [
    study.trials[int(row['number'])]
    for _, row in df_ok.head(N_TOP).iterrows()
]

param_cols = [c for c in df_ok.columns if c.startswith('params_')]
display(
    df_ok.head(N_TOP)[['number', 'value'] + param_cols]
    .rename(columns={'number': 'trial', 'value': 'top1_proxy_5ep',
                     **{c: c.replace('params_', '') for c in param_cols}})
    .reset_index(drop=True)
)
print(f'\n{len(df_ok)} trials completados — reentrenando top {N_TOP} con {N_EPOCHS} épocas')

In [ ]:
# ── Reentrenamiento top-N ─────────────────────────────────────────────────────

resultados = []

for rank, trial in enumerate(top_trials, start=1):
    p = trial.params
    print(f'\n{'='*60}')
    print(f'  Rank #{rank} — Trial {trial.number}  (proxy 5ep: {trial.value:.4f})')
    print(f'{'='*60}')

    _lda_alpha = None if p.get('lda_alpha_none', True) else p.get('lda_alpha')
    _lda_beta  = None if p.get('lda_beta_none',  True) else p.get('lda_beta')
    _norm      = None if p.get('norm', 'l2') == 'none' else p.get('norm', 'l2')

    pipe = PipelineMDLF(
        nombre_almacen          = ALMACEN,
        ttp_paths               = TTP_PATHS,
        od_matrices_subdirs     = OD_MATRICES_SUBDIRS,
        line_ap_path            = LINE_AP_PATH,
        split_date              = SPLIT_DATE,
        n_topics_cercanias      = p['n_topics'],
        lda_alpha               = _lda_alpha,
        lda_beta                = _lda_beta,
        lda_use_periods         = p['lda_use_periods'],
        lda_min_df              = p['lda_min_df'],
        lda_log_counts          = p['lda_log_counts'],
        lda_token_granularity   = p['lda_token_granularity'],
        lda_min_station_trips   = p['lda_min_station_trips'],
        n_heads                 = p['n_heads'],
        d_t                     = p['d_t'],
        cnn_channels            = p['cnn_channels'],
        kernel_size             = p['kernel_size'],
        optimizer               = p['optimizer'],
        lr                      = p['lr'],
        weight_decay            = p['weight_decay'],
        lr_scheduler            = p['lr_scheduler'],
        n_weeks                 = p['n_weeks'],
        include_prox            = p['include_prox'],
        td_mode                 = p.get('td_mode', 'context'),
        include_neighborhood_crowd = p['include_neighborhood_crowd'],
        norm                    = _norm,
        n_epochs                = N_EPOCHS,
        random_state            = 42,
    )

    pipe.run(run_name=f'mejores_combis_rank{rank}_t{trial.number}')
    pipe.attribution_by_group()

    resultados.append({
        'rank'          : rank,
        'trial'         : trial.number,
        'top1_proxy'    : trial.value,
        'top1_final'    : pipe.test_accuracy,
        'top5_final'    : pipe.test_accuracy5,
        'n_epochs'      : N_EPOCHS,
        'norm'          : _norm,
        'td_mode'       : p.get('td_mode', 'context'),
        'n_topics'      : p['n_topics'],
        'optimizer'     : p['optimizer'],
        'lr'            : round(p['lr'], 5),
        'n_weeks'       : p['n_weeks'],
        'include_prox'  : p['include_prox'],
        'incl_nbcrowd'  : p['include_neighborhood_crowd'],
    })

    print(f'  -> Top-1: {pipe.test_accuracy:.4f}  |  Top-5: {pipe.test_accuracy5:.4f}')

In [ ]:
# ── Comparativa final ─────────────────────────────────────────────────────────

df_res = pd.DataFrame(resultados).sort_values('top1_final', ascending=False).reset_index(drop=True)

print(f'\nRESULTADOS FINALES — {N_EPOCHS} épocas')
print('='*70)
display(df_res)

ganador = df_res.iloc[0]
print(f'\nMejor combinacion: Rank #{int(ganador["rank"])} (trial {int(ganador["trial"])})')
print(f'  Top-1 final : {ganador["top1_final"]:.4f}')
print(f'  Top-5 final : {ganador["top5_final"]:.4f}')
print(f'  Top-1 proxy : {ganador["top1_proxy"]:.4f}  (proxy 5ep Optuna)')

# Guardar en JSON
_out_path = _mdlf_dir / 'runs' / 'mejores_combis_resultados.json'
df_res.to_json(_out_path, orient='records', indent=2)
print(f'\nGuardado en: {_out_path}')